# DSAI 413 - Assignment 2  --  ColPali-RAG vs MedGemma on MIMIC-CXR

Headless-friendly Kaggle runbook (free T4, 16 GB).

Prerequisites (do these once, in the Kaggle UI):
1. **Settings -> Accelerator** = `GPU T4 x1`.
2. **Settings -> Internet** = On.
3. **Add data** -> `simhadrisadaram/mimic-cxr-dataset` (or push via `scripts/run_on_kaggle.py`).
4. **Add-ons -> Secrets** -> attach `HF_TOKEN` and `ANTHROPIC_API_KEY`.
5. (Once) Accept the MedGemma license at https://huggingface.co/google/medgemma-1.5-4b-it .

In [ ]:
# ----- repo to clone -----
REPO_URL    = 'https://github.com/MazenBasha/Assignment-2-Multimedia.git'
REPO_BRANCH = 'main'
# Override here if you want a different config (e.g. configs/config.yaml for full scale).
CFG         = 'configs/kaggle.yaml'
# -------------------------

## 1. Secrets, repo, dependencies

In [ ]:
import os
MISSING = []
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    try:
        os.environ['HF_TOKEN'] = sec.get_secret('HF_TOKEN')
        os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
        print('HF_TOKEN: OK')
    except Exception as e:
        MISSING.append('HF_TOKEN'); print(f'HF_TOKEN: MISSING ({e})')
    try:
        os.environ['ANTHROPIC_API_KEY'] = sec.get_secret('ANTHROPIC_API_KEY')
        print('ANTHROPIC_API_KEY: OK')
    except Exception as e:
        MISSING.append('ANTHROPIC_API_KEY'); print(f'ANTHROPIC_API_KEY: MISSING ({e})')
except ImportError:
    print('kaggle_secrets not available; assuming env vars are set externally.')

if 'HF_TOKEN' in MISSING:
    print('\n>>> Inference (ColPali index + MedGemma) WILL FAIL without HF_TOKEN. Attach it in Add-ons -> Secrets.')
if 'ANTHROPIC_API_KEY' in MISSING:
    print('\n>>> VQA dataset construction WILL FAIL without ANTHROPIC_API_KEY. Mode B will be skipped.')

In [ ]:
import os, subprocess
os.chdir('/kaggle/working')

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    gh = UserSecretsClient().get_secret('GH_TOKEN')
    if gh and url.startswith('https://github.com/'):
        url = url.replace('https://github.com/', f'https://{gh}@github.com/')
        print('Using GH_TOKEN for private repo auth.')
except Exception:
    pass

if not os.path.exists('cxr-rag'):
    subprocess.check_call(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, url, 'cxr-rag'])
else:
    subprocess.check_call(['git', '-C', 'cxr-rag', 'pull', '--ff-only'])
    print('repo updated')
os.chdir('/kaggle/working/cxr-rag')
print('cwd:', os.getcwd())
!ls

In [ ]:
# Kaggle's base image already has torch + transformers + Pillow + pandas + scikit-learn.
# We only need the extra retrieval / metric / LLM-judge libraries.
!pip install -q --no-deps colpali-engine==0.3.4 peft==0.13.0 einops==0.8.0
!pip install -q rouge-score==0.1.2 bert-score==0.3.13 nltk==3.9.1 sacrebleu==2.4.3
!pip install -q anthropic==0.39.0
!pip install -q bitsandbytes==0.43.3
print('deps installed')

In [ ]:
import torch, transformers, sys
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('transformers', transformers.__version__)
print('python', sys.version.split()[0])
assert torch.cuda.is_available(), 'No GPU. Set Settings -> Accelerator -> GPU T4 x1 and re-run.'

## 2. Data preparation

Kaggle mounts the dataset under `/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/`. `configs/kaggle.yaml` already points at that path; `data_prep.preprocess_kaggle` parses the per-patient row format into one row per study.

In [ ]:
!ls /kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset/ 2>&1 | head
import pathlib
p = pathlib.Path('/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset')
assert p.exists(), f'MIMIC dataset not attached. Right sidebar -> Add data -> simhadrisadaram/mimic-cxr-dataset.'

In [ ]:
!python -m data_prep.preprocess_kaggle --config $CFG

In [ ]:
!python -m data_prep.split --config $CFG

## 3. Build the VQA dataset (Claude Haiku as the LLM judge)

Soft-failing: if this errors (e.g. ANTHROPIC_API_KEY missing or credit exhausted), the notebook continues and Mode B is skipped at eval time.

In [ ]:
import subprocess
VQA_OK = True
try:
    subprocess.check_call(['python', '-m', 'data_prep.build_vqa_dataset', '--config', CFG])
    subprocess.check_call(['python', '-m', 'data_prep.split_vqa',          '--config', CFG])
except subprocess.CalledProcessError as e:
    VQA_OK = False
    print(f'VQA build failed: {e}. Continuing without Mode B.')

## 4. ColPali index over the training corpus

In [ ]:
!python -m retrieval.colpali_index --config $CFG

## 5. Generate predictions  --  Mode A (report) and Mode B (VQA)

In [ ]:
!python -m rag.run_mode_a --config $CFG --system both

In [ ]:
import subprocess
if VQA_OK:
    subprocess.check_call(['python', '-m', 'rag.run_mode_b', '--config', CFG, '--system', 'both'])
else:
    print('Skipping Mode B because VQA dataset build failed.')

## 6. Evaluate

In [ ]:
import subprocess, os
# Mode A always exists (we don't soft-fail it).
subprocess.check_call(['python', '-m', 'eval.metrics_report', '--config', CFG])
if VQA_OK and os.path.exists('outputs/mode_b_rag.jsonl'):
    subprocess.check_call(['python', '-m', 'eval.metrics_vqa', '--config', CFG])
else:
    print('Skipping eval.metrics_vqa  --  Mode B outputs not present.')
subprocess.check_call(['python', '-m', 'eval.qualitative_dump', '--config', CFG])

In [ ]:
import json, os, pandas as pd
OUT = '/kaggle/working/cxr-rag/outputs'
if os.path.exists(f'{OUT}/metrics_mode_a.json'):
    a = json.load(open(f'{OUT}/metrics_mode_a.json'))
    print('=== Mode A ===')
    display(pd.DataFrame({k: v for k, v in a.items() if k in ('rag','baseline')}).T)
if os.path.exists(f'{OUT}/metrics_mode_b.json'):
    b = json.load(open(f'{OUT}/metrics_mode_b.json'))
    print('=== Mode B ===')
    display(pd.DataFrame({k: v for k, v in b.items() if k in ('rag','baseline')}).T)

In [ ]:
# Stage outputs at /kaggle/working/ so scripts/run_on_kaggle.py's `kaggle kernels output`
# can find them at the kernel root regardless of the working dir.
import shutil, os, pathlib
SRC = '/kaggle/working/cxr-rag/outputs'
DST = '/kaggle/working'
if os.path.exists(SRC):
    for name in os.listdir(SRC):
        s = os.path.join(SRC, name)
        d = os.path.join(DST, name)
        if os.path.isfile(s):
            shutil.copy(s, d)
    print('staged outputs at /kaggle/working/:', sorted(p for p in os.listdir(DST) if p.startswith(('mode_','metrics_','qualitative_'))))

## 7. Persist outputs

Both `cxr-rag/outputs/` and `/kaggle/working/` will be kept. `scripts/run_on_kaggle.py --pull` downloads the staged files at the kernel root automatically.